# ImplementForge — Instructor Notebook
## COSE 278: Implementing Systems — Day 4

**This notebook is for the instructor only. Do not share with students.**

Use this to:
- Generate facilitator trace for any team after they submit their final summary
- Compare team outcomes side by side
- Get debrief questions keyed to each team's actual decisions
- Run the iteration comparison after second run


## ⚙️ CELL 0 — Setup
*Run once at start of day.*


In [ ]:
GITHUB_BASE = "https://raw.githubusercontent.com/rickghome/278game/main"
import urllib.request, os, pickle, copy

for filename in ["if_engine.py", "if_cards.py"]:
    urllib.request.urlretrieve(f"{GITHUB_BASE}/{filename}", filename)
    print(f"  ✅  {filename} loaded")

exec(open("if_engine.py").read())
exec(open("if_cards.py").read())

# Storage for all team game states
team_states = {}
team_states_v2 = {}
print("\n✅  Instructor notebook ready.")
print("    Paste each team's game_state.pkl path into CELL 1 as teams finish.")


## 📥 CELL 1 — Load Team State

Each team runs `save_game(game)` which writes `game_state.pkl` to their Colab session.
They share it with you via Google Drive or you download it from their session.

Run this cell once per team.


In [ ]:
# Set team name and path to their saved game_state.pkl
TEAM_NAME = "Team A"       # ← change this
PICKLE_PATH = "team_a_game_state.pkl"  # ← path to their file

try:
    with open(PICKLE_PATH, 'rb') as f:
        game = pickle.load(f)
    team_states[TEAM_NAME] = game
    print(f"✅  Loaded: {TEAM_NAME}")
    print(f"    Income: ${sum(game['income'].values()):,}")
    print(f"    Trust:  {game['trust_score']}/100")
    print(f"    Cards:  {game['cards_fired']}")
    print(f"    Seeds:  {game['seeds']}")
except FileNotFoundError:
    print(f"❌  File not found: {PICKLE_PATH}")
    print(f"    Ask the team to run save_game(game) and share the file.")


## ✏️ CELL 2 — Manual State Entry (alternative)

If you can't get the pickle file, teams can read you their CELL 15 output
and you reconstruct the key state manually.


In [ ]:
# Build game state manually from team's CELL 15 output
# Fill in what you see from their output

TEAM_NAME = "Team B"

manual_game = new_game_state(TEAM_NAME)
manual_game["environment"] = "government"   # ← from their output

manual_game["frame1"] = {
    "environment":        "government",
    "team_structure":     "siloed",
    "build_buy_configure":"configure",
    "primary_risk":       "delivery_speed",
    "data_architecture":  "shared_db",
    "coupling":           "medium",
}
manual_game["frame2"] = {
    "testing_coverage":    "minimal",
    "deployment_method":   "big_bang",
    "rollback_plan":       "none",
    "observability_level": "none",
    "change_owner":        "single_person",
}
manual_game["frame3"] = {
    "vendor_dependency":  "high",
    "fallback_strategy":  "none",
    "on_call_coverage":   "none",
    "incident_response":  "ad_hoc",
}

# From their cards output
manual_game["cards_fired"] = ["B1","B5","B6","L1","L3","L6","D1","S1","S2"]
manual_game["decisions"]   = {"B1":"c","B5":"b","B6":"a","L1":"b","L3":"b","L6":"b","D1":"c","S1":"n/a","S2":"b"}
manual_game["rationales"]  = {"B1":"we didn't have time to renegotiate","B5":"VP pushed us","B6":"nothing we could do","L1":"hotfix was fastest","L3":"we didn't know","L6":"competitive pressure","D1":"too expensive to split","S2":"selective was safer"}

# Seeds
manual_game["seeds"] = ["B1_gap_unresolved","B5_speed_skip","L6_uat_skip","D1_db_scaling"]

# Income + trust from their output
manual_game["income"] = {"phase1":380000,"phase2":210000,"phase3":145000,"phase4":88000}
manual_game["trust_score"] = 42

team_states[TEAM_NAME] = manual_game
print(f"✅  Manual state built for {TEAM_NAME}")
print(f"    Total income: ${sum(manual_game['income'].values()):,}")
print(f"    Trust: {manual_game['trust_score']}/100")


## 🔍 CELL 3 — Facilitator Trace

Run this for any team to get the full causal explanation.
Use it while debriefing that team with the room.


In [ ]:
DEBRIEF_TEAM = "Team A"   # ← change to team you're debriefing

if DEBRIEF_TEAM not in team_states:
    print(f"❌  {DEBRIEF_TEAM} not loaded. Run CELL 1 or CELL 2 first.")
else:
    print_facilitator_trace(team_states[DEBRIEF_TEAM])


## 📊 CELL 4 — All Teams Summary

Run after all teams have submitted. Shows the full leaderboard
with income, trust, and key config choices for comparison.


In [ ]:
if not team_states:
    print("❌  No teams loaded yet.")
else:
    sep = "━" * 60
    print(f"\n{sep}")
    print(f"IMPLEMENTFORGE — FINAL LEADERBOARD")
    print(sep)

    # Sort by total income
    ranked = sorted(team_states.items(),
                    key=lambda x: sum(x[1]['income'].values()), reverse=True)

    for rank, (name, g) in enumerate(ranked, 1):
        total = sum(g['income'].values())
        trust = g['trust_score']
        env   = g.get('environment','?')
        cards = g.get('cards_fired',[])
        seeds = g.get('seeds',[])
        f1    = g.get('frame1',{})
        f2    = g.get('frame2',{})
        print(f"\n  #{rank}  {name}")
        print(f"       Income: ${total:,}  |  Trust: {trust}/100  |  Env: {env}")
        print(f"       Deploy: {f2.get('deployment_method','?')}  |  Testing: {f2.get('testing_coverage','?')}  |  Data: {f1.get('data_architecture','?')}")
        print(f"       Cards fired: {len(cards)}  |  Seeds at end: {len(seeds)}")
        if seeds: print(f"       Active seeds: {seeds}")

    print(f"\n{sep}")
    print(f"\nDebrief anchor: 'Same events hit every team. Different outcomes. Why?'")


## 💬 CELL 5 — Debrief Questions

Suggested questions for room discussion after leaderboard reveal.
These are independent of any team's specific outcome.


In [ ]:
print("""
DEBRIEF QUESTIONS — POST LEADERBOARD REVEAL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Opening:
  "Look at the leaderboard. Same events hit every team.
   What produced the different outcomes?"

On config choices:
  "Which single config decision made the biggest difference?
   When you made that decision, did you know that?"

On seeds:
  "Teams that planted seeds in Phase 1 paid for them in Phase 3.
   What was the earliest moment any of those seeds could have been avoided?"

On trust vs income:
  "Some teams protected income but lost trust.
   Others lost income but protected trust.
   Which would you rather explain to a board?"

On the thriving team (if any):
  "Team X was thriving. What did it cost them to get there?
   Look at their capacity usage. Was that expensive resilience worth it?"

On the collapsing team (if any):
  "Team X collapsed. Trace it back —
   what was the first decision that made this outcome likely?"

On second iteration:
  "You've seen the consequences. What would you change?
   What does that tell you about what you should have known before you started?"

Closing:
  "Architecture is a hypothesis. Implementation is the test.
   What hypothesis did each team bring in — and what did the test reveal?"
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")


## 🔄 CELL 6 — Iteration Comparison

After the second iteration, load each team's v2 state and compare.


In [ ]:
# Load v2 state for a team
TEAM_NAME_V2 = "Team A"
PICKLE_PATH_V2 = "team_a_game_state_v2.pkl"

try:
    with open(PICKLE_PATH_V2, 'rb') as f:
        game_v2 = pickle.load(f)
    team_states_v2[TEAM_NAME_V2] = game_v2
    print(f"✅  Loaded v2: {TEAM_NAME_V2}")
except FileNotFoundError:
    print(f"❌  File not found: {PICKLE_PATH_V2}")


In [ ]:
# Compare iterations for a team
COMPARE_TEAM = "Team A"

if COMPARE_TEAM not in team_states:
    print(f"❌  {COMPARE_TEAM} v1 not loaded.")
elif COMPARE_TEAM not in team_states_v2:
    print(f"❌  {COMPARE_TEAM} v2 not loaded.")
else:
    print_iteration_comparison(team_states[COMPARE_TEAM], team_states_v2[COMPARE_TEAM])
